# Analyzing Promotion Differences Between Groups

There are two sections to this data exercise.  

Section A: Likelihood of Promotion

**chi-square tests of independence** and **binomial logistic regression**


* Are flexible work, diversity program membership, and store status associated with whether someone is promoted?

Section B: Time-to-Promotion Extension

**Survival Analysis**

* How long did it take for the employee to be promoted, and do flexible work, diversity program membership, or store status affect that timing?

Layer 1 — Promotion likelihood
* Chi-square tests
* Nested logistic regression
* Full logistic model


Layer 2 — Time-to-promotion
* Kaplan-Meier curves
* Log-rank tests
* Cox proportional hazards model

In [ ]:
# survival analysis
!pip install -q lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 4.5 MB/s eta 0:00:00


In [ ]:
# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# statistical tests
from scipy import stats

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# logistic regression diagnostics
from statsmodels.stats.outliers_influence import variance_inflation_factor

# sklearn metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines import CoxPHFitter

# display settings
pd.set_option("display.max_columns", None)

# plotting style
sns.set_style("whitegrid")

**Business / Research Framing**

Primary question:

Are flexible working status, diversity program membership, and store assignment associated with promotion likelihood?

Secondary question:

Do any apparent group differences remain after accounting for overlap among the other group variables?

In [ ]:
# load promotion dataset
url = "https://raw.githubusercontent.com/keithmcnulty/peopleanalytics-regression-book/master/data/promotion.csv"

promotion_df = pd.read_csv(url)

# basic inspection
print(promotion_df.shape)
print("\nColumns:")
print(promotion_df.columns)

promotion_df.head()

(1134, 5)

Columns:
Index(['diverse', 'flexible', 'store', 'promoted', 'year'], dtype='object')


,diverse,flexible,store,promoted,year
0,0,0,0,0,5
1,1,0,0,0,6
2,0,0,1,0,5
3,1,0,0,0,6
4,0,0,0,0,5


**Variable Classification**

| Variable   | Type                            | Role                                             |
| ---------- | ------------------------------- | ------------------------------------------------ |
| `promoted` | binary outcome, 0/1             | main dependent variable for promotion likelihood |
| `diverse`  | binary predictor, 0/1           | group/program membership predictor               |
| `flexible` | binary predictor, 0/1           | flexible work status predictor                   |
| `store`    | binary predictor, 0/1           | in-store role predictor                          |
| `year`     | numeric time/follow-up variable | duration variable for time-to-promotion analysis |


**Exploratory Data Analysis**

In [ ]:
# inspect data types
promotion_df.dtypes

,0
diverse,int64
flexible,int64
store,int64
promoted,int64
year,int64


In [ ]:
# check missing values
promotion_df.isnull().sum()

,0
diverse,0
flexible,0
store,0
promoted,0
year,0


In [ ]:
# summary statistics
promotion_df.describe()

,diverse,flexible,store,promoted,year
count,1134.000000,1134.000000,1134.000000,1134.000000,1134.000000
mean,0.381834,0.222222,0.514109,0.161376,5.534392
std,0.486051,0.415923,0.500021,0.368039,0.903147
min,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,5.000000
50%,0.000000,0.000000,1.000000,0.000000,6.000000
75%,1.000000,0.000000,1.000000,0.000000,6.000000
max,1.000000,1.000000,1.000000,1.000000,8.000000


In [ ]:
# overall promotion counts
promotion_df["promoted"].value_counts()

,count
promoted,
0,951
1,183


In [ ]:
# overall promotion rate
promotion_df["promoted"].value_counts(normalize=True)

,proportion
promoted,
0,0.838624
1,0.161376


In [ ]:
# counts for each binary predictor
for col in ["diverse", "flexible", "store"]:
    print(f"\n{col} counts")
    print(promotion_df[col].value_counts())
    print(f"\n{col} proportions")
    print(promotion_df[col].value_counts(normalize=True))


diverse counts
diverse
0    701
1    433
Name: count, dtype: int64

diverse proportions
diverse
0    0.618166
1    0.381834
Name: proportion, dtype: float64

flexible counts
flexible
0    882
1    252
Name: count, dtype: int64

flexible proportions
flexible
0    0.777778
1    0.222222
Name: proportion, dtype: float64

store counts
store
1    583
0    551
Name: count, dtype: int64

store proportions
store
1    0.514109
0    0.485891
Name: proportion, dtype: float64


In [ ]:
# promotion rates by each predictor
for col in ["diverse", "flexible", "store"]:
    print(f"\nPromotion rate by {col}")
    display(
        promotion_df
        .groupby(col)["promoted"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"sum": "promoted_count", "mean": "promotion_rate"})
    )


Promotion rate by diverse


,count,promoted_count,promotion_rate
diverse,,,
0,701,152,0.216833
1,433,31,0.071594



Promotion rate by flexible


,count,promoted_count,promotion_rate
flexible,,,
0,882,153,0.173469
1,252,30,0.119048



Promotion rate by store


,count,promoted_count,promotion_rate
store,,,
0,551,96,0.174229
1,583,87,0.149228


In [ ]:
# year distribution
promotion_df["year"].value_counts().sort_index()

,count
year,
1,1
2,2
3,20
4,107
5,354
6,556
7,75
8,19


In [ ]:
# promotion status by year
pd.crosstab(
    promotion_df["year"],
    promotion_df["promoted"],
    margins=True
)

promoted,0,1,All
year,,,
1,1,0,1
2,2,0,2
3,19,1,20
4,88,19,107
5,286,68,354
6,467,89,556
7,69,6,75
8,19,0,19
All,951,183,1134


In [ ]:
# overlap among predictors
pd.crosstab(
    promotion_df["flexible"],
    promotion_df["diverse"],
    margins=True
)

diverse,0,1,All
flexible,,,
0,607,275,882
1,94,158,252
All,701,433,1134


In [ ]:
pd.crosstab(
    promotion_df["flexible"],
    promotion_df["store"],
    margins=True
)

store,0,1,All
flexible,,,
0,442,440,882
1,109,143,252
All,551,583,1134


In [ ]:
pd.crosstab(
    promotion_df["diverse"],
    promotion_df["store"],
    margins=True
)

store,0,1,All
diverse,,,
0,347,354,701
1,204,229,433
All,551,583,1134


**Initial Group-Level Tests**

Use **chi-square tests of independence** for each group variable separately

Promotion vs Diversity Program

* H0: Promotion is independent of diversity program membership

* HA: Promotion differs by diversity program membership


Conclusion:

* Promotion outcome is statistically associated with diversity program membership.

* Employees in the diversity program appear to have a significantly lower promotion rate than employees not in the program.

Important caution:

* This is an unadjusted test. It does not prove that diversity program membership causes lower promotion likelihood. We still need the logistic regression and nested model analysis to see whether this relationship remains after accounting for flexible and store.

In [ ]:
# chi-square test: promoted vs diverse
diverse_table = pd.crosstab(
    promotion_df["diverse"],
    promotion_df["promoted"]
)

diverse_table

promoted,0,1
diverse,,
0,549,152
1,402,31


In [ ]:
# run chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(diverse_table)

expected_df = pd.DataFrame(
    expected,
    index=diverse_table.index,
    columns=diverse_table.columns
)

print("Chi-square Test: Promoted by Diverse")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df

Chi-square Test: Promoted by Diverse
Chi-square statistic: 40.6549
p-value: 0.000000
Degrees of freedom: 1


promoted,0,1
diverse,,
0,587.875661,113.124339
1,363.124339,69.875661


Use **chi-square tests of independence** for each group variable separately

Promotion vs Flexible Work

* H0: Promotion is independent of flexible working status

* HA: Promotion differs by flexible working status


Conclusion:

* Promotion outcome is statistically associated with flexible work status at the 5% significance level, but the evidence is weak/marginal.


* Employees on flexible work arrangements appear to have a lower promotion rate than those not on flexible work arrangements.

Caution:
* Since the p-value is close to 0.05, we should be cautious and see whether this relationship remains in the logistic regression after accounting for diverse and store

In [ ]:
# chi-square test: promoted vs flexible
flexible_table = pd.crosstab(
    promotion_df["flexible"],
    promotion_df["promoted"]
)

flexible_table

promoted,0,1
flexible,,
0,729,153
1,222,30


In [ ]:
# run chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(flexible_table)

expected_df = pd.DataFrame(
    expected,
    index=flexible_table.index,
    columns=flexible_table.columns
)

print("Chi-square Test: Promoted by Flexible")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df

Chi-square Test: Promoted by Flexible
Chi-square statistic: 3.8967
p-value: 0.048381
Degrees of freedom: 1


promoted,0,1
flexible,,
0,739.666667,142.333333
1,211.333333,40.666667


Use **chi-square tests of independence** for each group variable separately

Promotion vs Store Status

* H0: Promotion is independent of store status

* HA: Promotion differs by store status

Conclusion:
* There is no statistically significant evidence that promotion outcome is associated with store status in the unadjusted chi-square test.

In [ ]:
# chi-square test: promoted vs store
store_table = pd.crosstab(
    promotion_df["store"],
    promotion_df["promoted"]
)

store_table

promoted,0,1
store,,
0,455,96
1,496,87


In [ ]:
# run chi-square test of independence
chi2, p, dof, expected = stats.chi2_contingency(store_table)

expected_df = pd.DataFrame(
    expected,
    index=store_table.index,
    columns=store_table.columns
)

print("Chi-square Test: Promoted by Store")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df

Chi-square Test: Promoted by Store
Chi-square statistic: 1.1301
p-value: 0.287761
Degrees of freedom: 1


promoted,0,1
store,,
0,462.082011,88.917989
1,488.917989,94.082011


 **Assumption Checks for Chi-Square Tests**


 ## Chi-Square Assumption Checks

For each chi-square test, the expected cell counts were reviewed to ensure that the chi-square approximation was appropriate. All expected cell counts were comfortably above the common minimum threshold of 5. Therefore, the chi-square tests for `diverse`, `flexible`, and `store` are appropriate.

The observations are also treated as independent individuals, and each variable is coded into mutually exclusive categories. These assumptions support the validity of the chi-square tests.

In [ ]:
# check expected cell counts for chi-square tests
def check_chi_square_expected_counts(df, predictor, outcome="promoted"):
    table = pd.crosstab(df[predictor], df[outcome])
    chi2, p, dof, expected = stats.chi2_contingency(table)

    expected_df = pd.DataFrame(
        expected,
        index=table.index,
        columns=table.columns
    )

    print(f"Expected counts: {outcome} by {predictor}")
    display(expected_df)

    print(f"Minimum expected count: {expected_df.min().min():.2f}")
    print("-" * 50)

for col in ["diverse", "flexible", "store"]:
    check_chi_square_expected_counts(promotion_df, col)

Expected counts: promoted by diverse


promoted,0,1
diverse,,
0,587.875661,113.124339
1,363.124339,69.875661


Minimum expected count: 69.88
--------------------------------------------------
Expected counts: promoted by flexible


promoted,0,1
flexible,,
0,739.666667,142.333333
1,211.333333,40.666667


Minimum expected count: 40.67
--------------------------------------------------
Expected counts: promoted by store


promoted,0,1
store,,
0,462.082011,88.917989
1,488.917989,94.082011


Minimum expected count: 88.92
--------------------------------------------------


**Baseline Logistic Regression Models**

promoted ∼ flexible

promoted ∼diverse

promoted ∼store

Purpose: Estimate the direction, size, and significance of each factor’s unadjusted association with promotion likelihood.


The baseline logistic results are consistent with the chi-square tests.
* flexible is statistically significant and negatively associated with promotion.
* diverse is statistically significant and negatively associated with promotion.
* store is negatively associated with promotion, but not statistically significant.

promoted ∼ flexible
* In the unadjusted logistic model, employees who worked on a flexible/part-time program for at least six months had approximately 36% lower odds of promotion compared with employees who did not, and this result was statistically significant.





promoted ∼diverse
* In the unadjusted logistic model, employees who were members of the diversity program had approximately 72% lower odds of promotion compared with non-members, and this result was statistically significant.

promoted ∼store
* Employees who joined in a retail store position had approximately 17% lower odds of promotion compared with non-store employees, but this result was not statistically significant.

The most important caveat:

* These are unadjusted models, so we should not yet conclude that flexible work or diversity program membership independently explains promotion likelihood. The next step is the nested/full logistic model to see whether these associations remain after accounting for overlap among the predictors.

In [ ]:
# baseline logistic model: promoted ~ flexible
model_flexible = smf.logit(
    formula="promoted ~ flexible",
    data=promotion_df
).fit()

print(model_flexible.summary())

Optimization terminated successfully.
         Current function value: 0.439940
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               promoted   No. Observations:                 1134
Model:                          Logit   Df Residuals:                     1132
Method:                           MLE   Df Model:                            1
Date:                Mon, 18 May 2026   Pseudo R-squ.:                0.004534
Time:                        22:42:27   Log-Likelihood:                -498.89
converged:                       True   LL-Null:                       -501.16
Covariance Type:            nonrobust   LLR p-value:                   0.03302
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.5612      0.089    -17.557      0.000      -1.736      -1.387
flexible      -0.4402      0.

In [ ]:
# odds ratios for flexible model
np.exp(model_flexible.params)

,0
Intercept,0.209877
flexible,0.643879


In [ ]:
# baseline logistic model: promoted ~ diverse
model_diverse = smf.logit(
    formula="promoted ~ diverse",
    data=promotion_df
).fit()

print(model_diverse.summary())

Optimization terminated successfully.
         Current function value: 0.421635
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               promoted   No. Observations:                 1134
Model:                          Logit   Df Residuals:                     1132
Method:                           MLE   Df Model:                            1
Date:                Mon, 18 May 2026   Pseudo R-squ.:                 0.04595
Time:                        22:42:32   Log-Likelihood:                -478.13
converged:                       True   LL-Null:                       -501.16
Covariance Type:            nonrobust   LLR p-value:                 1.147e-11
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.2842      0.092    -14.012      0.000      -1.464      -1.105
diverse       -1.2782      0.

In [ ]:
# odds ratios for diverse model
np.exp(model_diverse.params)

,0
Intercept,0.276867
diverse,0.278525


In [ ]:
# baseline logistic model: promoted ~ store
model_store = smf.logit(
    formula="promoted ~ store",
    data=promotion_df
).fit()

print(model_store.summary())

Optimization terminated successfully.
         Current function value: 0.441367
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               promoted   No. Observations:                 1134
Model:                          Logit   Df Residuals:                     1132
Method:                           MLE   Df Model:                            1
Date:                Mon, 18 May 2026   Pseudo R-squ.:                0.001305
Time:                        22:42:35   Log-Likelihood:                -500.51
converged:                       True   LL-Null:                       -501.16
Covariance Type:            nonrobust   LLR p-value:                    0.2528
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.5559      0.112    -13.854      0.000      -1.776      -1.336
store         -0.1847      0.

In [ ]:
# odds ratios for store model
np.exp(model_store.params)

,0
Intercept,0.210989
store,0.831338


**Nested Logistic Regression Strategy**


Model 1 — Flexible Only
* promoted∼flexible

Model 2 — Flexible + Diversity
* promoted∼flexible+diverse

Model 3 — Flexible + Diversity + Store
* promoted∼flexible+diverse+store

Purpose:
* Determine whether the apparent association between flexible work and promotion changes after accounting for diversity program membership and store status.

**Nested Model Comparison**

Key questions:

* Does adding diverse significantly improve the model?
* Does adding store significantly improve the model?
* Does the flexible coefficient shrink, grow, or lose significance after controls?
* Are observed differences direct associations, or partly explained by overlap among groups?

**Interpretation**



Once diverse is added, the flexible coefficient shrinks from about -0.44 to about -0.08 and becomes clearly non-significant. That suggests the initial flexible-work association was largely explained by overlap with diversity program membership.

Diverse remains large, negative, and statistically significant across the adjusted models. In the full model, the odds ratio is about 0.284, meaning diversity program members have about 71.6% lower odds of promotion, holding flexible and store constant.

Adding store does not materially improve the model.

The pseudo-R barely changes, AIC slightly worsens, and BIC worsens more. So from a parsimony standpoint, the flexible + diverse model may be preferable.

**Key conclusion**:

* The apparent flexible-work promotion gap appears to be explained largely by overlap with diversity program membership. Diversity program membership is the strongest and most stable predictor of promotion outcomes in these models. Store status does not add meaningful explanatory value.

In [ ]:
# nested logistic model 1: flexible only
model_1_flexible = smf.logit(
    formula="promoted ~ flexible",
    data=promotion_df
).fit()

print(model_1_flexible.summary())

Optimization terminated successfully.
         Current function value: 0.439940
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               promoted   No. Observations:                 1134
Model:                          Logit   Df Residuals:                     1132
Method:                           MLE   Df Model:                            1
Date:                Mon, 18 May 2026   Pseudo R-squ.:                0.004534
Time:                        22:57:52   Log-Likelihood:                -498.89
converged:                       True   LL-Null:                       -501.16
Covariance Type:            nonrobust   LLR p-value:                   0.03302
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.5612      0.089    -17.557      0.000      -1.736      -1.387
flexible      -0.4402      0.

In [ ]:
# nested logistic model 2: flexible + diverse
model_2_flexible_diverse = smf.logit(
    formula="promoted ~ flexible + diverse",
    data=promotion_df
).fit()

print(model_2_flexible_diverse.summary())

Optimization terminated successfully.
         Current function value: 0.421573
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               promoted   No. Observations:                 1134
Model:                          Logit   Df Residuals:                     1131
Method:                           MLE   Df Model:                            2
Date:                Mon, 18 May 2026   Pseudo R-squ.:                 0.04609
Time:                        22:57:53   Log-Likelihood:                -478.06
converged:                       True   LL-Null:                       -501.16
Covariance Type:            nonrobust   LLR p-value:                 9.276e-11
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.2732      0.096    -13.250      0.000      -1.462      -1.085
flexible      -0.0839      0.

In [ ]:
# nested logistic model 3: flexible + diverse + store
model_3_full = smf.logit(
    formula="promoted ~ flexible + diverse + store",
    data=promotion_df
).fit()

print(model_3_full.summary())

Optimization terminated successfully.
         Current function value: 0.421143
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               promoted   No. Observations:                 1134
Model:                          Logit   Df Residuals:                     1130
Method:                           MLE   Df Model:                            3
Date:                Mon, 18 May 2026   Pseudo R-squ.:                 0.04707
Time:                        22:57:56   Log-Likelihood:                -477.58
converged:                       True   LL-Null:                       -501.16
Covariance Type:            nonrobust   LLR p-value:                 3.188e-10
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.1950      0.123     -9.687      0.000      -1.437      -0.953
flexible      -0.0674      0.

In [ ]:
# compare nested logistic models
nested_comparison = pd.DataFrame({
    "Model": [
        "Flexible only",
        "Flexible + Diverse",
        "Flexible + Diverse + Store"
    ],
    "Pseudo_R2": [
        model_1_flexible.prsquared,
        model_2_flexible_diverse.prsquared,
        model_3_full.prsquared
    ],
    "AIC": [
        model_1_flexible.aic,
        model_2_flexible_diverse.aic,
        model_3_full.aic
    ],
    "BIC": [
        model_1_flexible.bic,
        model_2_flexible_diverse.bic,
        model_3_full.bic
    ],
    "LogLik": [
        model_1_flexible.llf,
        model_2_flexible_diverse.llf,
        model_3_full.llf
    ]
})

nested_comparison

,Model,Pseudo_R2,AIC,BIC,LogLik
0,Flexible only,0.004534,1001.784141,1011.851154,-498.892071
1,Flexible + Diverse,0.046095,962.127009,977.227529,-478.063505
2,Flexible + Diverse + Store,0.047067,963.152788,983.286814,-477.576394


In [ ]:
# odds ratios for full model
np.exp(model_3_full.params)

,0
Intercept,0.302689
flexible,0.934813
diverse,0.283673
store,0.849665


**Full Logistic Regression Model**

Final baseline model:

* promoted ∼ flexible + diverse + store

Purpose:
* Estimate the adjusted association between each group variable and promotion likelihood while holding the others constant.


Interpretation:

* Holding flexible work and store status constant, employees in the diversity program had approximately 72% lower odds of promotion than employees not in the diversity program. Flexible work and store status were not statistically significant predictors of promotion after accounting for the other variables.

**Logistic Regression Diagnostics**

Interpretation Framework

For each predictor, interpret:

* unadjusted promotion rate difference
* unadjusted odds ratio
* adjusted odds ratio from full model
* whether the coefficient changes across nested models
* whether the result is statistically and practically meaningful

Key distinction:

A raw group difference may disappear after adjustment if it was explained by overlap with another group variable.

**Model convergence**

The model converged successfully, so there is no obvious estimation failure.

When we say the logistic regression model converged, we mean the estimation algorithm successfully found a stable set of coefficient estimates.

Logistic regression estimates coefficients using an iterative optimization process. It starts with initial guesses for the coefficients, checks how well they fit the data, adjusts them, and repeats until the improvement becomes very small.

In [ ]:
# confirm model convergence
print("Model converged:", model_3_full.mle_retvals["converged"])

Model converged: True


In [ ]:
# events per predictor check
n_promoted = promotion_df["promoted"].sum()
n_predictors = 3

events_per_predictor = n_promoted / n_predictors

print(f"Number promoted: {n_promoted}")
print(f"Number of predictors: {n_predictors}")
print(f"Events per predictor: {events_per_predictor:.2f}")

Number promoted: 183
Number of predictors: 3
Events per predictor: 61.00


In [ ]:
# VIF check for full logistic model predictors
X_vif = promotion_df[["flexible", "diverse", "store"]].astype(float)

vif_df = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif_df

,variable,VIF
0,flexible,1.349777
1,diverse,1.466564
2,store,1.313920


In [ ]:
# fitted probabilities from full logistic model
promotion_df["predicted_prob_full"] = model_3_full.predict(promotion_df)

promotion_df["predicted_prob_full"].describe()

,predicted_prob_full
count,1134.000000
mean,0.161376
std,0.071627
min,0.063846
25%,0.074303
50%,0.204572
75%,0.232357
max,0.232357


In [ ]:
# check for near-perfect predicted probabilities
print("Predicted probabilities near 0:")
print((promotion_df["predicted_prob_full"] < 0.01).sum())

print("\nPredicted probabilities near 1:")
print((promotion_df["predicted_prob_full"] > 0.99).sum())

Predicted probabilities near 0:
0

Predicted probabilities near 1:
0


In [ ]:
# ROC-AUC for full logistic model
auc = roc_auc_score(
    promotion_df["promoted"],
    promotion_df["predicted_prob_full"]
)

print(f"ROC-AUC: {auc:.4f}")

ROC-AUC: 0.6407


In [ ]:
# compare model fit / parsimony from nested models
nested_comparison

,Model,Pseudo_R2,AIC,BIC,LogLik
0,Flexible only,0.004534,1001.784141,1011.851154,-498.892071
1,Flexible + Diverse,0.046095,962.127009,977.227529,-478.063505
2,Flexible + Diverse + Store,0.047067,963.152788,983.286814,-477.576394


**Section 1 conclusion**

The logistic regression diagnostics are acceptable. The model converged, sample size is sufficient, multicollinearity is low, and there is no evidence of separation. Substantively, the promotion-likelihood analysis shows that diversity program membership is the strongest adjusted predictor, while flexible work loses significance once diversity status is included. Store status does not add meaningful explanatory value. The model is useful for inference, but its modest ROC-AUC suggests that promotion likelihood is likely influenced by other unobserved factors not included in this dataset.


**Predictive strength**

* The predicted probabilities range only from about 6% to 23%, and ROC-AUC is:

**Survival-analysis structure**

**Create survival variables**

**Kaplan-Meier curves**

**Log-rank tests**

**Cox proportional hazards model**